In [1]:
import logging
import os
import sys

from pathlib import Path
from dotenv import load_dotenv


load_dotenv()

print("Execution Started...")

try:
    project_root_directory = Path(__file__).resolve().parent.parent
except NameError:
    project_root_directory = Path.cwd().parent

print(project_root_directory)

def setup_logger(file_path: str | None = None, level = logging.INFO) -> logging.Logger:
    print("Setup Logger Started...")
    try:

        print("Checking if any previous logging handler exists...")
        root_handler = logging.getLogger()
        print(f"Root Handlers: {root_handler.handlers}")
        if root_handler.handlers:
            return logging.getLogger(__name__)

        print(f"Initial file_path value: {file_path}")
        if file_path is None:
            file_path = os.getenv('LOG_FILE_PATH')

        if not file_path:
            raise ValueError("No log file path provided")

        print(f"File Path: {file_path}")

        log_path = Path(file_path)

        if not log_path.is_absolute():
            log_path = project_root_directory / log_path

        print(f"Absolute Log Path: {log_path}")

        log_path.parent.mkdir(parents=True, exist_ok=True)

        print("Configuring logger...")

        logging.basicConfig(
            level=level,
            format="%(asctime)s | %(funcName)s | %(levelname)s | %(message)s",
            handlers=[
                logging.FileHandler(log_path, encoding='utf-8'),
                logging.StreamHandler(sys.stdout)
            ],
            force=True
        )

        return logging.getLogger(__name__)
    
    except Exception as error:
        print(f"Configuring setup_logger failed due to {error}")
        raise

logger = setup_logger()

Execution Started...
c:\dvd-rental-modern-data-platform
Setup Logger Started...
Checking if any previous logging handler exists...
Root Handlers: []
Initial file_path value: None
File Path: logs/dvd_rental_interface.log
Absolute Log Path: c:\dvd-rental-modern-data-platform\logs\dvd_rental_interface.log
Configuring logger...


In [11]:
def main():
    logger.info('Pipeline Started...')
    validate_settings()

main()

2026-09-27 12:25:18,367 | main | INFO | Pipeline Started...
2026-09-27 12:25:18,368 | validate_settings | INFO | No missing required settings


In [10]:
AWS_REGION = os.getenv('AWS_REGION')
S3_BRONZE_BUCKET = os.getenv('S3_BRONZE_BUCKET')
RAW_DATA_PATH = os.getenv('RAW_DATA_PATH')

def validate_settings() -> None:

    try:
        required_settings = {
            'AWS_REGION': AWS_REGION,
            'S3_BRONZE_BUCKET': S3_BRONZE_BUCKET,
            'RAW_DATA_PATH': RAW_DATA_PATH
        }

        missing_setting = [name for name, value in required_settings.items() if not value]

        if missing_setting:
            logger.warning(f"Missing required settings: {missing_setting}")
            raise ValueError(f"missing required setting: {missing_setting}")
        else:
            logger.info("No missing required settings")

    except Exception as error:
        logger.error(f"Setiings validation failed due to {error}")
        raise
    

In [1]:
from dataclasses import dataclass
from datetime import datetime, timezone


@dataclass(frozen=True)
class BatchContext:
    batch_id: str
    ingestion_timestamp: datetime
    ingestion_date: str


def create_batch_context():

    ingestion_timestamp = datetime.now(timezone.utc)
    ingestion_date = ingestion_timestamp.strftime('%Y%m%d')
    batch_id = f"{ingestion_timestamp.strftime('%Y%m%d_%H%M%S')}_dvdrental"

    return BatchContext(
        batch_id=batch_id,
        ingestion_timestamp=ingestion_timestamp,
        ingestion_date=ingestion_date
    )

create_batch_context()

BatchContext(batch_id='20260928_171851_dvdrental', ingestion_timestamp=datetime.datetime(2026, 9, 28, 17, 18, 51, 385608, tzinfo=datetime.timezone.utc), ingestion_date='20260928')

In [2]:
import boto3
import sys

from botocore.exceptions import ClientError
from pathlib import Path

PROJECT_ROOT_DIR = Path.cwd().parent

if str(PROJECT_ROOT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR))

from src.utils.setup_logger import logger


class S3Client:
    def __init__(self, region_name: str):
        self.client = boto3.client('s3', region_name=region_name)

    def upload_file_to_s3(self, local_path: Path, bucket: str, s3_key: str) -> None:

        file_path = Path(local_path)

        if not file_path.is_file():
            raise FileNotFoundError(f"No file located at {file_path}")

        try:
            logger.info(f'Uploading file to S3: {file_path}')

            self.client.upload_file(str(file_path), bucket, s3_key, ExtraArgs = {"ContentType": "text/csv"})

            logger.info(f"File successfully uploaded to S3://{bucket}/{s3_key}")
            
        except ClientError as error:
            logger.error(f"Failed to upload file {file_path.name} due to: {error}")
            raise

if __name__ == "__main__":
    s3_client = S3Client(region_name='ap-south-1')

    s3_client.upload_file_to_s3(
        local_path=Path('C:/dvd-rental-modern-data-platform/data/raw/address.csv'),
        bucket='dvd-rental-modern-data-platform-bronze',
        s3_key='raw/address.csv'
    )
        

2026-10-01 10:03:29,351 | load | INFO | Found credentials in shared credentials file: ~/.aws/credentials
2026-10-01 10:03:29,781 | upload_file_to_s3 | INFO | Uploading file to S3: C:\dvd-rental-modern-data-platform\data\raw\address.csv


S3UploadFailedError: Failed to upload C:\dvd-rental-modern-data-platform\data\raw\address.csv to dvd-rental-modern-data-platform-bronze/raw/address.csv: An error occurred (AccessDenied) when calling the PutObject operation: User: arn:aws:iam::851382689558:user/ecommerce-sales-etl-and-analysis-user is not authorized to perform: s3:PutObject on resource: "arn:aws:s3:::dvd-rental-modern-data-platform-bronze/raw/address.csv" because no identity-based policy allows the s3:PutObject action